In [1]:
import os
import json
import numpy as np
import tensorflow as tf

from tensorflow.keras.applications.resnet50 import preprocess_input
from tensorflow.keras.preprocessing.text import tokenizer_from_json
from tensorflow.keras.preprocessing.image import load_img, img_to_array

2024-12-15 13:54:01.337420: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2024-12-15 13:54:01.413230: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2024-12-15 13:54:01.484042: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:485] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2024-12-15 13:54:01.549157: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:8454] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2024-12-15 13:54:01.567523: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1452] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2024-12-15 13:54:01.683779: I tensorflow/core/platform/cpu_feature_gu

In [2]:
from gensim.models import KeyedVectors
from sklearn.metrics.pairwise import cosine_similarity

vector_words_model = KeyedVectors.load_word2vec_format('./models/cc.ru.300.vec', binary=False)

In [3]:
def get_vector(word):
    try:
        return vector_words_model[word]
    except KeyError:
        return None

def calculate_similarity(tag1, tag2):
    vec1 = get_vector(tag1)
    vec2 = get_vector(tag2)
    if vec1 is not None and vec2 is not None:
        return cosine_similarity([vec1], [vec2])[0][0]
    else:
        return 0.0

In [4]:
class Model:
    def __init__(self, mod_path, tok_path):
        self.model_path = mod_path
        self.tokenizer_path = tok_path

    def switch(self, other_model):
        self.model_path = other_model.model_path
        self.tokenizer_path = other_model.tokenizer_path
        self.load()

    def load(self):
        self.model = tf.keras.models.load_model(self.model_path)
        with open(self.tokenizer_path, 'r', encoding='utf-8') as f:
            tokenizer_json = f.read()
            self.tokenizer = tokenizer_from_json(tokenizer_json)
        self.vocab_size = len(self.tokenizer.word_index)+1


In [5]:
MODEL_28_09_24 = Model("./models/28_09_2024.keras", "./models/28_09_2024.json")
MODEL_30_09_24 = Model("./models/30_09_2024.keras", "./models/28_09_2024.json")
MODEL_04_10_24 = Model("./models/04_10_2024.keras", "./models/04_10_2024.json")
MODEL_06_10_24 = Model("./models/06_10_2024.keras", "./models/06_10_2024.json")
MODEL_08_10_24 = Model("./models/08_10_2024.keras", "./models/08_10_2024.json")
MODEL_07_12_24 = Model("./models/07_12_2024.keras", "./models/07_12_2024.json")
MODEL_08_12_24 = Model("./models/08_12_2024.keras", "./models/08_12_2024.json")
MODEL_08_12_24V2 = Model("./models/08_12_2024v2.keras", "./models/08_12_2024.json")
MODEL_11_12_24 = Model("./models/11_12_2024.keras", "./models/11_12_2024.json")
MODEL_15_12_24 = Model("./models/15_12_2024.keras", "./models/15_12_2024.json")

MAX_LENGTH = 32  # максимальное количество токенов на выходе

In [6]:
def find_image(dir, num):
    return next(filter(lambda f: f.startswith(f"{num}."), os.listdir(dir)))

def generate_desc(model, tokenizer, photo):
    sequence = [tokenizer.word_index['start']]

    sequence_input = np.array([sequence[-1]])
    result = np.argmax(model.predict([photo, sequence_input], verbose=0), axis=-1)
    return [tokenizer.index_word.get(token) for token in filter(None, result[0])]

In [7]:
current_model = Model("", "")
current_model.switch(MODEL_15_12_24)

In [8]:
dir_path = "./train/"
for num in range(35, 47):
    # Загрузка изображения
    photo_path = dir_path+find_image(dir_path, num)
    photo = load_img(photo_path, target_size=(512, 512))
    photo = img_to_array(photo)
    photo = np.expand_dims(photo, axis=0)  # Добавляем размерность batch
    photo = preprocess_input(photo)

    description = generate_desc(current_model.model, current_model.tokenizer, photo)
    print(f"{num} ---", description)

2024-12-15 13:59:03.114267: W external/local_tsl/tsl/framework/cpu_allocator_impl.cc:83] Allocation of 16777216 exceeds 10% of free system memory.
2024-12-15 13:59:03.152154: W external/local_tsl/tsl/framework/cpu_allocator_impl.cc:83] Allocation of 17040384 exceeds 10% of free system memory.
2024-12-15 13:59:03.158850: W external/local_tsl/tsl/framework/cpu_allocator_impl.cc:83] Allocation of 16777216 exceeds 10% of free system memory.
2024-12-15 13:59:03.174407: W external/local_tsl/tsl/framework/cpu_allocator_impl.cc:83] Allocation of 16777216 exceeds 10% of free system memory.
2024-12-15 13:59:03.197506: W external/local_tsl/tsl/framework/cpu_allocator_impl.cc:83] Allocation of 16777216 exceeds 10% of free system memory.


35 --- ['аниме', 'девушка', 'девушка', 'костюм', 'костюм', 'костюм', 'текст']
36 --- ['аниме', 'девушка', 'девушка', 'грудь', 'румянец', 'румянец', 'румянец']
37 --- ['рисунок', 'аниме', 'девушка', 'грудь', 'соски', 'грудь', 'грудь', 'бантик', 'бантик']
38 --- ['аниме', 'девушка', 'девушка', 'ушки', 'ушки', 'ушки', 'ушки', 'ушки']
39 --- ['аниме', 'девушка', 'девушка', 'грудь', 'очки', 'очки', 'улыбка', 'чулки']
40 --- ['аниме', 'девушка', 'девушка', 'грудь', 'грудь', 'грудь', 'румянец', 'чулки', 'бантик']
41 --- ['аниме', 'девушка', 'девушка', 'зеленые', 'перчатки', 'перчатки', 'перчатки', 'перчатки']
42 --- ['мем', 'голова', 'мальчик', 'текст', 'текст', 'текст']
43 --- ['деньги', 'рисунок', 'желтый', 'белый']
44 --- ['рисунок', 'рыцарь', 'оружие', 'оружие', 'оружие', 'оружие', 'оружие']
45 --- ['рисунок', 'аниме', 'улыбка', 'улыбка', 'улыбка']
46 --- ['город', 'здания', 'здания', 'деревья']


In [11]:
with open("./train/descriptions.json", "r", encoding="utf-8") as file:
    real_descriptions = json.load(file)

In [9]:
class Mark:
    def __init__(self, real, predict):
        self.real = real
        self.predict = predict

        predicted_set = set(predict)
        real_set = set(real)

        self.common = real_set.intersection(predicted_set)
        self.extra = predicted_set.difference(real_set)
        self.unused = real_set.difference(predicted_set)

        try:
            self.extra_unused_similarity_grades=[]
            for e_tag in self.extra:
                self.extra_unused_similarity_grades.append((e_tag, max([calculate_similarity(e_tag, u_tag) for u_tag in self.unused])))
        except ValueError:
            pass
        
        try:
            self.extra_common_similarity_grades=[]
            for e_tag in self.extra:
                self.extra_common_similarity_grades.append((e_tag, max([calculate_similarity(e_tag, c_tag) for c_tag in self.common])))
        except ValueError:
            pass

    def summary(self):
       print("Количество совпавших тегов:", len(self.common))
       print("Количество добавленных тегов:", len(self.extra))
       print("Количество потерянных тегов:", len(self.unused))
       print("Насколько лишние теги подходят вместо неиспользованных:", self.extra_unused_similarity_grades)
       print("Насколько лишние теги подходят, как дополнения к совпавшим:", self.extra_common_similarity_grades)

    def mark(self):
        common_weight = 0.2
        extra_common_weight = 0.4
        extra_unused_weight = 0.4
        result=0
        add1=0
        add2=0
        result+=(len(self.common)/len(self.real))*common_weight
        try:
            result+=(sum(map(lambda pair: pair[1], self.extra_common_similarity_grades))/len(self.extra_common_similarity_grades))*extra_common_weight
        except:
            if result!=common_weight:
                add1=0*extra_common_weight
            else:
                add1=1*extra_common_weight
        try:
            result+=(sum(map(lambda pair: pair[1], self.extra_unused_similarity_grades))/len(self.extra_unused_similarity_grades))*extra_unused_weight
        except:
            if result!=common_weight:
                add2=0*extra_unused_weight
            else:
                add2=1*extra_unused_weight
        return result+add1+add2

In [12]:
dir_path = "./train/"

def get_image_desc(number):
    photo_path = dir_path+find_image(dir_path, number)
    photo = load_img(photo_path, target_size=(512, 512))
    photo = img_to_array(photo)
    photo = np.expand_dims(photo, axis=0)  # Добавляем размерность batch
    photo = preprocess_input(photo)

    # Генерация описания
    return generate_desc(current_model.model, current_model.tokenizer, photo)

def get_model_mark(tests_count, verbose=False):
    global_mark = 0
    for i in range(1, tests_count+1):
        test_number = str(i)
        descriptions = get_image_desc(test_number)
        real = real_descriptions[test_number]
        mark = Mark(real, descriptions)
        res = mark.mark()
        if verbose:
            print(f"============ test {i} ============")
            print("hyp:", descriptions)
            print("ref:", real)
            mark.summary()
            print("res_mark:", res*100)
        global_mark+=res
    return global_mark/tests_count*100

get_model_mark(20, verbose=True)



============ test 1 ============
hyp: ['пейзаж', 'поле', 'лес', 'трава', 'трава', 'трава']
ref: ['поле', 'трава', 'цветы', 'небо', 'облака']
Количество совпавших тегов: 2
Количество добавленных тегов: 2
Количество потерянных тегов: 3
Насколько лишние теги подходят вместо неиспользованных: [('лес', 0.41607192), ('пейзаж', 0.38934386)]
Насколько лишние теги подходят, как дополнения к совпавшим: [('лес', 0.3931506), ('пейзаж', 0.31796965)]
res_mark: 38.330720543861396
============ test 2 ============
hyp: ['игра', 'горы', 'гора', 'скалы', 'скалы']
ref: ['планета', 'луна', 'человек', 'силуэт', 'ночь', 'облака', 'туман', 'вода', 'река', 'трава', 'горы', 'скалы']
Количество совпавших тегов: 2
Количество добавленных тегов: 2
Количество потерянных тегов: 10
Насколько лишние теги подходят вместо неиспользованных: [('игра', 0.3936434), ('гора', 0.5303647)]
Насколько лишние теги подходят, как дополнения к совпавшим: [('игра', 0.12083269), ('гора', 0.6007084)]
res_mark: 36.24431764086088
=========

27.748752569253483